In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Cubic calibration coefficients from thermocouple_calibration.ipynb
# V_mV = a3*t^3 + a2*t^2 + a1*t + a0,  t = (T_K - T_MEAN) / T_STD
CAL_COEFFS = np.array([-5.371838e+01, 1.037465e+02, 1961.945128, 1227.4515])
CAL_T_MEAN = 398.27    # K
CAL_T_STD  = 193.13    # K
CAL_T_MIN_K = 73.15    # K  (-200 °C)
CAL_T_MAX_K = 573.15   # K  (300 °C)

def voltage_to_temperature(voltage_V):
    """Convert thermocouple voltage (V) to temperature (°C) via cubic calibration."""
    voltage_mV = voltage_V * 1000.0
    shifted = CAL_COEFFS.copy()
    shifted[-1] -= voltage_mV

    roots = np.roots(shifted)
    real_roots = roots[np.abs(roots.imag) < 1e-6].real
    T_candidates = real_roots * CAL_T_STD + CAL_T_MEAN
    in_domain = T_candidates[(T_candidates >= CAL_T_MIN_K) & (T_candidates <= CAL_T_MAX_K)]

    if len(in_domain) == 0:
        return np.nan
    return float(in_domain[0]) - 273.15

df = pd.read_csv('nikitas_experimets.csv')
df['temperature_C'] = df['thermo_volt'].apply(voltage_to_temperature)
df['capacitance_pF'] = df['capacitance_value'] * 1e12

# Filter out outliers where capacitance > 100 pF (1e-10 F)
df = df[df['capacitance_value'] <= 1e-10]

# Filter out points below -30 °C
df = df[df['temperature_C'] >= -30]
df.head()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(df['temperature_C'], df['capacitance_pF'], s=10, alpha=0.6, label='Data')

# Linear best fit
mask = df['temperature_C'].notna() & df['capacitance_pF'].notna()
m, b = np.polyfit(df.loc[mask, 'temperature_C'], df.loc[mask, 'capacitance_pF'], 1)
T_fit = np.linspace(df['temperature_C'].min(), df['temperature_C'].max(), 200)
ax.plot(T_fit, m * T_fit + b, 'r-', linewidth=2, label=f'Linear fit: C = {m:.4f}T + {b:.2f} pF')

ax.set_xlabel('Temperature (°C)', fontsize=13)
ax.set_ylabel('Capacitance (pF)', fontsize=13)
ax.set_title('Capacitance vs Temperature', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Thermal Expansion Coefficient from Capacitance Data

The thiourea crystal is **glued to one plate** and acts as a spacer — it is not a dielectric between the plates. The dielectric is air ($\varepsilon_0$), and the plate separation equals the crystal length along the b-axis:

$$C(T) = \frac{\varepsilon_0 A}{L(T)} = \frac{\varepsilon_0 A}{L_0(1 + \alpha \Delta T)} = \frac{C_0}{1 + \alpha \Delta T}$$

Linearizing for small $\alpha \Delta T$: $\; C(T) \approx C_0(1 - \alpha \Delta T)$, so the **fractional** slope is:

$$\frac{1}{C_0}\frac{dC}{dT} = -\alpha$$

The no-sample capacitor has similar geometry, so its **fractional** slope captures apparatus drift. Subtracting:

$$\alpha = -\left[\frac{1}{C_0^{(s)}}\frac{dC^{(s)}}{dT} - \frac{1}{C_0^{(n)}}\frac{dC^{(n)}}{dT}\right]$$

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

# Load both datasets
df_sample = pd.read_csv('final_data/sample_capacitances.csv')
df_nosample = pd.read_csv('final_data/no_sample_capacitances.csv')

# Convert to pF
df_sample['capacitance_pF'] = df_sample['capacitance_value'] * 1e12
df_nosample['capacitance_pF'] = df_nosample['capacitance_value'] * 1e12

# Reliable temperature ranges
T_MIN_SAMPLE, T_MAX = -30, 25
T_MIN_NOSAMPLE = -15

df_s = df_sample[(df_sample['temperature_C'] >= T_MIN_SAMPLE) & (df_sample['temperature_C'] <= T_MAX)].copy()
df_n = df_nosample[(df_nosample['temperature_C'] >= T_MIN_NOSAMPLE) & (df_nosample['temperature_C'] <= T_MAX)].copy()

# Remove capacitance outliers (> 3 sigma from rolling median)
for label, d in [('sample', df_s), ('no_sample', df_n)]:
    med = d['capacitance_pF'].median()
    std = d['capacitance_pF'].std()
    mask = (d['capacitance_pF'] > med - 3*std) & (d['capacitance_pF'] < med + 3*std)
    if label == 'sample':
        df_s = d[mask].copy()
    else:
        df_n = d[mask].copy()

print(f"Sample:    {len(df_s)} pts, T = [{df_s['temperature_C'].min():.1f}, {df_s['temperature_C'].max():.1f}] °C")
print(f"No sample: {len(df_n)} pts, T = [{df_n['temperature_C'].min():.1f}, {df_n['temperature_C'].max():.1f}] °C")

# Plot both datasets
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.scatter(df_s['temperature_C'], df_s['capacitance_pF'], s=10, alpha=0.6, color='tab:blue')
ax1.set_xlabel('Temperature (°C)')
ax1.set_ylabel('Capacitance (pF)')
ax1.set_title('With Sample')
ax1.grid(True, alpha=0.3)

ax2.scatter(df_n['temperature_C'], df_n['capacitance_pF'], s=10, alpha=0.6, color='tab:orange')
ax2.set_xlabel('Temperature (°C)')
ax2.set_ylabel('Capacitance (pF)')
ax2.set_title('Without Sample (baseline)')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ── Fractional baseline correction & single overall fit ──
from scipy.optimize import curve_fit

T_OVERLAP_MIN, T_OVERLAP_MAX = -15, 25
T_REF = 25.0

df_s_overlap = df_s[(df_s['temperature_C'] >= T_OVERLAP_MIN) & (df_s['temperature_C'] <= T_OVERLAP_MAX)]
df_n_overlap = df_n[(df_n['temperature_C'] >= T_OVERLAP_MIN) & (df_n['temperature_C'] <= T_OVERLAP_MAX)]

# Linear fits to raw data
m_s, b_s = np.polyfit(df_s_overlap['temperature_C'], df_s_overlap['capacitance_pF'], 1)
m_n, b_n = np.polyfit(df_n_overlap['temperature_C'], df_n_overlap['capacitance_pF'], 1)

C0_s = m_s * T_REF + b_s
C0_n = m_n * T_REF + b_n

# Fractional correction: divide out apparatus drift
T_s = df_s_overlap['temperature_C'].values
C_s = df_s_overlap['capacitance_pF'].values
C_ns_at_Ts = m_n * T_s + b_n
C_corrected = C_s * C0_n / C_ns_at_Ts

# Overall α from fractional slopes (linear)
alpha_linear = -(m_s/C0_s - m_n/C0_n)

# Nonlinear fit: C(T) = C₀ / (1 + α·(T - T_ref))
def cap_model(T, C0, alpha):
    return C0 / (1 + alpha * (T - T_REF))

popt, pcov = curve_fit(cap_model, T_s, C_corrected, p0=[C0_s, 1e-4], maxfev=10000)
C0_fit, alpha_fit = popt
perr = np.sqrt(np.diag(pcov))

print("=" * 60)
print("THERMAL EXPANSION COEFFICIENT (b-axis, thiourea)")
print("=" * 60)
print(f"  C₀ (sample, at {T_REF}°C):    {C0_s:.2f} pF")
print(f"  C₀ (no-sample, at {T_REF}°C): {C0_n:.2f} pF")
print(f"  Fractional slope (sample):    {m_s/C0_s:.6f} /°C")
print(f"  Fractional slope (no-sample): {m_n/C0_n:.6f} /°C")
print(f"")
print(f"  α (linear estimate):    {alpha_linear:.4e} /K")
print(f"  α (nonlinear fit):      {alpha_fit:.4e} ± {perr[1]:.4e} /K")
print(f"  C₀ (nonlinear fit):     {C0_fit:.3f} ± {perr[0]:.3f} pF")
print("=" * 60)

# Plot
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(T_s, C_corrected, s=15, alpha=0.6, color='tab:green', label='Baseline-corrected data')

T_plot = np.linspace(T_OVERLAP_MIN, T_OVERLAP_MAX, 300)
ax.plot(T_plot, cap_model(T_plot, *popt), 'k-', lw=2,
        label=f'Fit: C₀/(1+α·ΔT),  α = {alpha_fit:.2e} ± {perr[1]:.1e} /K')

ax.set_xlabel('Temperature (°C)', fontsize=13)
ax.set_ylabel('Corrected Capacitance (pF)', fontsize=13)
ax.set_title('Thermal Expansion — Baseline-Corrected (fractional)', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()